# 🚀 THSA-2B: Pre-Training & QAT Distillation Pipeline (Google Colab)
### Bilingual English/Bangla & World History ShareGPT Conversational Training

This notebook trains the **THSA-2B / THSA-350M** on-device hybrid neural model (16 State Short-Conv + 8 GQA Attention + Ternary 1.58-bit BitNet FFN) using **Quantization-Aware Training (QAT)** and **LoRA** parameter-efficient fine-tuning on Google Colab GPUs (T4 / V100 / A100).

In [ ]:
# Step 1: Check GPU Acceleration
!nvidia-smi

In [ ]:
# Step 2: Clone the Official Repository
!git clone https://github.com/tpxplatfrom-afk/SS_module_BD.git
%cd SS_module_BD/ss_bangladesh_nano_android_module/THSA-2B\ V1

In [ ]:
# Step 3: Install Required Dependencies
!pip install -q torch transformers datasets sentencepiece accelerate bitsandbytes

In [ ]:
# Step 4: Generate / Verify Bilingual ShareGPT Dataset (90% Train / 10% Test)
!python data/build_bilingual_sharegpt_dataset.py

In [ ]:
# Step 5: Inspect Dataset Samples
import json
with open('data/train_sharegpt.jsonl', 'r', encoding='utf-8') as f:
    sample = json.loads(f.readline())
    print('Topic:', sample.get('topic'))
    print('Turns:', len(sample.get('conversations', [])))
    print('First Human Prompt:', sample['conversations'][0]['value'][:200], '...')

## 🏋️ Step 6: Train Model with QAT Distillation + LoRA
We use **LoRA (Low-Rank Adaptation)** on top of the BitNet ternary layers for ultra-fast convergence and minimal VRAM consumption on Colab.

In [ ]:
# Option A: Train Fast 350M Proxy Model (~3-5 mins on T4 GPU)
!python training/train_qat.py \
    --config training/config/proxy_350m_config.json \
    --train_data data/train_sharegpt.jsonl \
    --test_data data/test_sharegpt.jsonl \
    --output checkpoints/thsa_trained_model.pt \
    --epochs 5 \
    --batch_size 2 \
    --lr 3e-4 \
    --use_lora \
    --lora_r 16

In [ ]:
# Option B (Uncomment for Full 2B Model Training):
# !python training/train_qat.py \
#     --config training/config/thsa_2b_config.json \
#     --train_data data/train_sharegpt.jsonl \
#     --test_data data/test_sharegpt.jsonl \
#     --output checkpoints/thsa_2b_trained_model.pt \
#     --epochs 10 \
#     --batch_size 1 \
#     --lr 1e-4 \
#     --use_lora \
#     --lora_r 32

## 📦 Step 7: Export Trained PyTorch Checkpoint to .nano Binary Distribution
Converts the trained PyTorch state dict into a 64-byte aligned, zero-copy memory mapped `.nano` binary file.

In [ ]:
!python tools/export_to_nano.py \
    --config training/config/proxy_350m_config.json \
    --checkpoint checkpoints/thsa_trained_model.pt \
    --output models/model_trained.nano

## 🧪 Step 8: Verify Exported .nano Model with Native C++ Verification Suite

In [ ]:
!python tools/inspect_nano_binary.py models/model_trained.nano

In [ ]:
# Download the final trained .nano package to your local machine / Android app
from google.colab import files
files.download('models/model_trained.nano')